In [1]:
import requests
import pandas as pd
import time
from tqdm import tqdm

In [2]:
API_KEY = "84287db0-182f-4d1b-9d8e-3c2047664fe4"

In [3]:
BASE_URL = "https://content.guardianapis.com/search"

params = {
    "api-key": API_KEY,
    "section": "business",
    "from-date": "2019-01-01",
    "to-date": "2024-12-31",
    "show-fields": "bodyText,headline,byline",
    "show-tags": "type,keyword,tone",
    "page-size": 50,   # maximum allowed
    "page": 1
}

In [4]:
response = requests.get(BASE_URL, params=params)

print("Status code:", response.status_code)
print(response.url)

data = response.json()

Status code: 200
https://content.guardianapis.com/search?api-key=84287db0-182f-4d1b-9d8e-3c2047664fe4&section=business&from-date=2019-01-01&to-date=2024-12-31&show-fields=bodyText%2Cheadline%2Cbyline&show-tags=type%2Ckeyword%2Ctone&page-size=50&page=1


In [5]:
all_articles = []

# First request to get total pages
response = requests.get(BASE_URL, params=params)
data = response.json()

total_pages = data["response"]["pages"]
print(f"Total pages in 2019-2024 in business section: {total_pages}")

for page in tqdm(range(1, total_pages + 1)):
    params["page"] = page
    
    response = requests.get(BASE_URL, params=params)
    data = response.json()
    
    articles = data["response"]["results"]
    
    for article in articles:
        fields = article.get("fields", {})
        
        all_articles.append({
            "webTitle": article.get("webTitle"),
            "sectionName": article.get("sectionName"),
            "webPublicationDate": article.get("webPublicationDate"),
            "headline": fields.get("headline"),
            "byline": fields.get("byline"),
            "bodyText": fields.get("bodyText"),
            "tags": article.get("tags"),
            "webUrl": article.get("webUrl")
        })
    
    time.sleep(1)

Total pages in 2019-2024 in business section: 461


100%|████████████████████████████████████████████████████████████████████████████████| 461/461 [09:37<00:00,  1.25s/it]


In [6]:
df = pd.DataFrame(all_articles)

print("Total articles pulled:", len(df))

Total articles pulled: 23027


In [7]:
df.head()

,webTitle,sectionName,webPublicationDate,headline,byline,bodyText,tags,webUrl
0,Great Guinness heist: thieves stole truck carr...,Business,2024-12-31T18:18:27Z,Great Guinness heist: thieves stole truck carr...,Rob Davies and Helena Horton,"In the days leading up to Christmas, stout-lov...","[{'id': 'business/hospitality-industry', 'type...",https://www.theguardian.com/business/2024/dec/...
1,FTSE 100 rallies by 5.7% in 2024 in ‘a year of...,Business,2024-12-31T16:25:35Z,FTSE 100 rallies by 5.7% in 2024 in ‘a year of...,Graeme Wearden,A late PS: The three major U.S. stock indices ...,"[{'id': 'business/business', 'type': 'keyword'...",https://www.theguardian.com/business/live/2024...
2,Russia winds down gas supply to Europe via Ukr...,Business,2024-12-31T15:18:52Z,Russia winds down gas supply to Europe via Ukr...,Jillian Ambrose Energy correspondent,Europe will receive the last Russian gas sent ...,"[{'id': 'business/gas', 'type': 'keyword', 'se...",https://www.theguardian.com/business/2024/dec/...
3,FTSE records strongest annual gain since 2021 ...,Business,2024-12-31T15:16:25Z,FTSE records strongest annual gain since 2021 ...,Graeme Wearden,The UK’s blue-chip stock index has recorded it...,"[{'id': 'business/ftse', 'type': 'keyword', 's...",https://www.theguardian.com/business/2024/dec/...
4,Green light: the boss of GB Railfreight with a...,Business,2024-12-31T13:00:46Z,Green light: the boss of GB Railfreight with a...,Jack Simpson,"Travel north on the east coast mainline, the m...","[{'id': 'business/rail-industry', 'type': 'key...",https://www.theguardian.com/business/2024/dec/...


In [8]:
row = df.iloc[0]

for col in df.columns:
    print(f"{col}: {row[col]}")

webTitle: Great Guinness heist: thieves stole truck carrying 35,000 pints
sectionName: Business
webPublicationDate: 2024-12-31T18:18:27Z
headline: Great Guinness heist: thieves stole truck carrying 35,000 pints
byline: Rob Davies and Helena Horton
bodyText: In the days leading up to Christmas, stout-lovers were left reeling from a nationwide shortage of Guinness so severe that some pubs were forced to ration pints of the “black stuff” as taps began to run dry. Supermarkets remain at risk of running out due to customers’ stockpiling, according to reports, while the maker of the popular stout, Diageo, has even sent for back-up Guinness reserves from Ireland. Now it can be revealed that criminals appear to have gone to even greater lengths to beat the drought, with a heist that exacerbated the nationwide shortage. A truck carrying 400 50-litre kegs of the Irish stout – equivalent to 35,200 pints – disappeared from a depot in the Midlands in mid-December, the Guardian can reveal. It is und

In [9]:
df.describe(include="all")

,webTitle,sectionName,webPublicationDate,headline,byline,bodyText,tags,webUrl
count,23027,23027,23027,23027,22871,23027,23027,23027
unique,23011,1,22596,23011,2176,23023,22113,23027
top,UK inflation: which goods and services have ch...,Business,2019-06-26T16:00:19Z,UK inflation: which goods and services have ch...,Sarah Butler,,"[{'id': 'business/retail', 'type': 'keyword', ...",https://www.theguardian.com/business/2019/jan/...
freq,9,23027,5,9,1416,5,48,1


In [10]:
print("Total Authors:", df["byline"].nunique())

Total Authors: 2176


In [11]:
print("Most Frequent Authors")
print(df["byline"].value_counts().head(10))
print("*" * 15)
print("Least Frequent Authors")
print(df["byline"].value_counts().tail(10))

Most Frequent Authors
byline
Sarah Butler                                  1416
Jasper Jolly                                  1235
Graeme Wearden                                1116
Julia Kollewe                                  984
Nils Pratley                                   915
Phillip Inman                                  846
Mark Sweney                                    845
Richard Partington Economics correspondent     785
Kalyeena Makortoff Banking correspondent       658
Rob Davies                                     621
Name: count, dtype: int64
***************
Least Frequent Authors
byline
Zoe Wood City reporter                                 1
Sarah Boseley and Rob Davies                           1
Larry Elliott, Richard Partington and Jasper Jolly     1
Sarah Butler and Press Association                     1
Sam Levin in Windsor, California                       1
Hannah Ellis-Petersen South east Asia correspondent    1
Katharine Murphy and Lisa Cox                  

In [12]:
len(df[df["byline"].str.contains(" and ", na=False)]["byline"].value_counts())

1261

In [13]:
df[df["byline"].str.contains(" and ", na=False)]["byline"].value_counts().head(5)

byline
Guardian staff and agencies                            56
Elias Visontay Transport and urban affairs reporter    52
Jasper Jolly and agencies                              38
Guardian staff and agency                              30
Staff and agencies                                     26
Name: count, dtype: int64

In [14]:
df[df["byline"].str.contains(" and ", na=False)]["byline"].value_counts().tail(5)

byline
Jasper Jolly and Justin McCurry in Tokyo    1
Sarah Butler and Angela Monaghan            1
Dominic Rushe and Phillip Inman             1
Sarah Boseley and Rob Davies                1
Joanna Partridge  and Gwyn Topham           1
Name: count, dtype: int64

In [15]:
df["byline"].isna().sum()

np.int64(156)

In [16]:
(df["byline"] == "").sum()

np.int64(76)

In [17]:
df["year"] = pd.to_datetime(df["webPublicationDate"]).dt.year

author_by_year = df.groupby("year")["byline"].nunique()
print(author_by_year)

year
2019    484
2020    545
2021    554
2022    626
2023    594
2024    555
Name: byline, dtype: int64


In [18]:
df.to_csv("guardian_business.csv", index=False)